In [0]:
# Databricks notebook source
from pyspark.sql.functions import sum as _sum, count, round as _round, col

df = spark.table("learning.silver.sales_orders")

# Calculate net revenue per order: (quantity * unit_price) - discount
df_with_revenue = df.withColumn(
    "revenue",
    (col("quantity") * col("unit_price")) - col("discount")
)

# Aggregate by state and category — common gold-layer business view
df_gold = df_with_revenue.groupBy("state", "category").agg(
    _sum("revenue").alias("total_revenue"),
    _sum("quantity").alias("total_quantity_sold"),
    count("*").alias("order_count"),
    _round(_sum("revenue") / count("*"), 2).alias("avg_order_value")
)

df_gold.write.format("delta").mode("overwrite").saveAsTable("learning.gold.sales_summary")

display(spark.table("learning.gold.sales_summary"))

In [0]:
display(df.select("status").distinct())

In [0]:
display(df_gold.select("category").distinct())